<a href="https://colab.research.google.com/github/gcalanch/DMA-Caras/blob/main/PreProcesamiento_GC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Paso 1: Instalación de dependencias

In [8]:
!pip install face_alignment scikit-learn opencv-python-headless --quiet

Paso 2: Montar Google Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Paso 3: Preprocesamiento de imágenes

In [ ]:
import os
import cv2
import numpy as np
import face_alignment
from pathlib import Path
from tqdm import tqdm

input_dir = "/content/drive/MyDrive/DMA/Caras"
output_dir = "/content/drive/MyDrive/DMA/Caras_Recortadas"
img_size = (60, 60)

# Inicializar detector de landmarks
fa = face_alignment.FaceAlignment(2, flip_input=False, device='cpu')


def detectar_y_procesar(img_path):
    try:
        img = cv2.imread(img_path)
        if img is None:
            return None
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        preds = fa.get_landmarks(gray)
        if preds is None:
            return None
        landmarks = preds[0]

        # Centrar y alinear cara con landmarks (ojos)
        left_eye = np.mean(landmarks[36:42], axis=0)
        right_eye = np.mean(landmarks[42:48], axis=0)
        dY = right_eye[1] - left_eye[1]
        dX = right_eye[0] - left_eye[0]
        angle = np.degrees(np.arctan2(dY, dX))

        eyes_center = tuple(((left_eye + right_eye) / 2).astype(int))
        M = cv2.getRotationMatrix2D(eyes_center, angle, 1.0)
        aligned = cv2.warpAffine(gray, M, (gray.shape[1], gray.shape[0]), flags=cv2.INTER_CUBIC)

        # Detectar cara con OpenCV
        face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
        faces = face_cascade.detectMultiScale(aligned, scaleFactor=1.1, minNeighbors=3)

        if len(faces) == 0:
            return None

        x, y, w, h = faces[0]
        face_img = aligned[y:y+h, x:x+w]

        # Redimensionar, normalizar e igualar histograma
        resized = cv2.resize(face_img, img_size)
        normalized = cv2.equalizeHist(resized)
        return normalized
    except:
        return None

# Procesar todas las imágenes
for person_folder in tqdm(os.listdir(input_dir)):
    person_path = os.path.join(input_dir, person_folder)
    if not os.path.isdir(person_path):
        continue

    output_person_path = os.path.join(output_dir, person_folder)
    os.makedirs(output_person_path, exist_ok=True)

    for img_file in os.listdir(person_path):
        img_path = os.path.join(person_path, img_file)
        processed_img = detectar_y_procesar(img_path)
        if processed_img is not None:
            save_path = os.path.join(output_person_path, img_file)
            cv2.imwrite(save_path, processed_img)


  0%|          | 0/14 [00:00<?, ?it/s]

Opcion para requerir menos RAM

In [1]:
import os
import subprocess
from tqdm import tqdm

input_dir = "/content/drive/MyDrive/DMA/Caras"
output_dir = "/content/drive/MyDrive/DMA/Caras_Recortadas"

script_path = "/content/process_one_image.py"

for person_folder in tqdm(os.listdir(input_dir)):
    person_path = os.path.join(input_dir, person_folder)
    if not os.path.isdir(person_path):
        continue

    output_person_path = os.path.join(output_dir, person_folder)
    os.makedirs(output_person_path, exist_ok=True)

    for img_file in os.listdir(person_path):
        img_path = os.path.join(person_path, img_file)
        output_path = os.path.join(output_person_path, img_file)

        # Saltar si ya existe
        if os.path.exists(output_path):
            continue

        # Llama al script externo para procesar una imagen
        subprocess.run([
            "python3", script_path,
            "--input", img_path,
            "--output", output_path
        ])



100%|██████████| 14/14 [00:48<00:00,  3.46s/it]


In [3]:
import cv2
import numpy as np
import face_alignment
import argparse
import os

parser = argparse.ArgumentParser()
parser.add_argument('--input', required=True, help='Ruta de la imagen')
parser.add_argument('--output', required=True, help='Ruta de salida')
args = parser.parse_args()

img_path = args.input
output_path = args.output
img_size = (60, 60)

try:
    # Cargar imagen
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None:
        exit()

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Inicializar modelo solo para una imagen
    fa = face_alignment.FaceAlignment(face_alignment.LandmarksType._2D, flip_input=False, device='cpu')
    preds = fa.get_landmarks(gray)
    if preds is None:
        exit()

    landmarks = preds[0]
    left_eye = np.mean(landmarks[36:42], axis=0)
    right_eye = np.mean(landmarks[42:48], axis=0)

    # Alinear rostro
    dY = right_eye[1] - left_eye[1]
    dX = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dY, dX))
    eyes_center = tuple(((left_eye + right_eye) / 2).astype(int))
    M = cv2.getRotationMatrix2D(eyes_center, angle, 1.0)
    aligned = cv2.warpAffine(gray, M, (gray.shape[1], gray.shape[0]), flags=cv2.INTER_LINEAR)

    # Detectar rostro
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(aligned, scaleFactor=1.1, minNeighbors=3)
    if len(faces) == 0:
        exit()

    x, y, w, h = faces[0]
    face_img = aligned[y:y+h, x:x+w]

    # Redimensionar y normalizar
    resized = cv2.resize(face_img, img_size, interpolation=cv2.INTER_AREA)
    normalized = cv2.equalizeHist(resized)

    # Guardar imagen
    cv2.imwrite(output_path, normalized)

except Exception as e:
    pass  # Silenciar errores por estabilidad


usage: colab_kernel_launcher.py [-h] --input INPUT --output OUTPUT
colab_kernel_launcher.py: error: the following arguments are required: --input, --output


SystemExit: 2

Paso 4: Clustering con DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def load_processed_images(base_dir):
    images = []
    labels = []
    paths = []
    for person in os.listdir(base_dir):
        person_dir = os.path.join(base_dir, person)
        for fname in os.listdir(person_dir):
            img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
            images.append(img.flatten() / 255.0)
            labels.append(person)
            paths.append(os.path.join(person_dir, fname))
    return np.array(images), labels, paths

X, y_labels, img_paths = load_processed_images(output_dir)

# Reducir dimensionalidad para clustering
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X)

# Clustering DBSCAN
clustering = DBSCAN(eps=2.5, min_samples=3).fit(X_pca)
labels = clustering.labels_

# Visualizar con t-SNE
from sklearn.manifold import TSNE
X_embedded = TSNE(n_components=2).fit_transform(X_pca)

plt.figure(figsize=(10,6))
plt.scatter(X_embedded[:,0], X_embedded[:,1], c=labels, cmap='tab10', s=10)
plt.title("Clustering DBSCAN con t-SNE")
plt.colorbar()
plt.show()


Paso 5: Selección de parámetros con ISOMAP + GridSearchCV

In [ ]:
from sklearn.model_selection import RepeatedKFold, GridSearchCV
from sklearn.metrics import pairwise_distances
from sklearn.pipeline import Pipeline
from sklearn.manifold import Isomap
from sklearn.neighbors import KNeighborsRegressor

# Usamos la distancia como objetivo artificial
def compute_avg_distance(X_train, X_test):
    dists = pairwise_distances(X_test, X_train)
    return np.mean(np.min(dists, axis=1))

# Dummy regressor para ajustar distancias como error
class DistanceScorer:
    def __init__(self, X):
        self.X = X

    def score(self, isomap):
        X_trans = isomap.fit_transform(self.X)
        rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)
        errors = []
        for train_idx, test_idx in rkf.split(X_trans):
            X_train, X_test = X_trans[train_idx], X_trans[test_idx]
            dist = compute_avg_distance(X_train, X_test)
            errors.append(dist)
        return np.mean(errors)

# Grid Search
param_grid = {
    'n_neighbors': [3, 5, 10],
    'n_components': [2, 5, 10, 20]
}

results = []
for n_n in param_grid['n_neighbors']:
    for n_c in param_grid['n_components']:
        iso = Isomap(n_neighbors=n_n, n_components=n_c)
        scorer = DistanceScorer(X)
        error = scorer.score(iso)
        results.append((n_n, n_c, error))
        print(f"Vecinos: {n_n}, Componentes: {n_c}, Error medio: {error:.4f}")

best_params = min(results, key=lambda x: x[2])
print(f"\n📌 Mejor configuración: vecinos={best_params[0]}, componentes={best_params[1]} con error {best_params[2]:.4f}")
